# 14. Boosting (AdaBoost & Gradient Boosting)

Compare two boosting methods:
1. **AdaBoost** — reweight misclassified samples; next weak learner focuses on hard examples
2. **Gradient Boosting** — add trees that fit the remaining error (gradient of the loss)

Compare to single tree / bagging / RF results already saved.
Same preprocessing: `scale=False`. Fixed `n_estimators=100`.

In [1]:
import sys
sys.path.append('..')

from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier

from src.data import load_adult
from src.preprocess import split_features_target, make_model_pipeline
from src.evaluate import cross_val_scores, evaluate_on_test, save_results

X, y = split_features_target(load_adult('train'))
X_test, y_test = split_features_target(load_adult('test'))
print(X.shape, y.mean())

(32561, 12) 0.2408095574460244


In [2]:
def run_and_save(name, model, extra):
    pipe = make_model_pipeline(model, scale=False)
    cv = cross_val_scores(pipe, X, y)
    print('CV:', cv)

    pipe2 = make_model_pipeline(clone(model), scale=False)
    test = evaluate_on_test(pipe2, X, y, X_test, y_test)
    print('Test:', test['metrics'])
    print('Confusion matrix:', test['confusion_matrix'])

    path = save_results(
        name,
        cv,
        test,
        extra=extra,
    )
    print('Saved to:', path)

In [3]:
# ADA Boosting

ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=100,
    random_state=42
)

run_and_save(
    'adaboost',
    ada,
    extra={
        'base': 'DecisionTreeClassifier max_depth=1',
        'n_estimators': 100,
    }
)

CV: {'accuracy': {'mean': 0.8551640623496912, 'std': 0.0027643529066186273}, 'balanced_accuracy': {'mean': 0.7618407448789536, 'std': 0.006378133063916412}, 'precision': {'mean': 0.7608303208908795, 'std': 0.01208316255906249}, 'recall': {'mean': 0.5818125577190723, 'std': 0.015871318201230902}, 'f1': {'mean': 0.659147315854897, 'std': 0.008946309477209159}, 'roc_auc': {'mean': 0.9091380069432263, 'std': 0.0014058949577147198}, 'average_precision': {'mean': 0.7824175582703199, 'std': 0.005847341415238956}}
Test: {'accuracy': 0.855352865303114, 'balanced_accuracy': 0.7600174887574513, 'precision': 0.7514333895446881, 'recall': 0.5793031721268851, 'f1': 0.6542357950374394, 'roc_auc': 0.9083741749348302, 'average_precision': 0.7763298859997836}
Confusion matrix: [[11698, 737], [1618, 2228]]
Saved to: /Users/kang/Desktop/ML project/Estimating-Income/results/adaboost.json


In [4]:
# Gradient Boosting

gb = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
)

run_and_save(
    'gradientboost',
    gb,
    extra={
        'n_estimators': 100,
        'max_depth': 3,
        'learning_rate': 0.1,
    }
)

CV: {'accuracy': {'mean': 0.8659745430703515, 'std': 0.003210703851311022}, 'balanced_accuracy': {'mean': 0.7767553863188893, 'std': 0.004973143076781206}, 'precision': {'mean': 0.7897748820702466, 'std': 0.013159632428968242}, 'recall': {'mean': 0.6046434587219209, 'std': 0.010964898616023984}, 'f1': {'mean': 0.6847959941601985, 'std': 0.007636415514017604}, 'roc_auc': {'mean': 0.9216991310188977, 'std': 0.0017682966036391948}, 'average_precision': {'mean': 0.8156510282789954, 'std': 0.005121988501222193}}
Test: {'accuracy': 0.8697254468398747, 'balanced_accuracy': 0.7791243953738849, 'precision': 0.7926705123854768, 'recall': 0.6073842953718148, 'f1': 0.6877668187840424, 'roc_auc': 0.9204109627995896, 'average_precision': 0.8111222717330135}
Confusion matrix: [[11824, 611], [1510, 2336]]
Saved to: /Users/kang/Desktop/ML project/Estimating-Income/results/gradientboost.json
